# Chatbot con RAG + LLM sobre información tabular
## Procesamiento de Lenguaje Natural — Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey · Prof. Luis Eduardo Falcón Morales

**Programa:** Maestría en Inteligencia Artificial Aplicada - MNA  
**Materia:** Procesamiento de Lenguaje Natural  
**Profesor:** Dr. Luis Eduardo Falcón Morales  
**Equipo:** Equipo 32  
**Nombre(s):** <br>Mario Arturo Salinas Rodríguez — A01696938<br>
Manuel Díaz Borrego — A01421216<br>
Ricardo Alejandro Corral García — A01561579<br>
Ana Karen Estupiñán Pacheco - A01796893<br>

**Fecha:** 27 de junio de 2026


---
### Objetivo
Implementar un **chatbot con RAG (Retrieval-Augmented Generation)** capaz de responder preguntas técnicas
basándose en dos hojas de referencia (*cheat-sheets*): `python_cheatsheet.pdf` y `ml_cheatsheet.pdf`.

El **reto central** es la extracción y recuperación de **información tabular** y su impacto en la
**alucinación** del modelo. Comparamos **tres** estrategias de extracción y mostramos por qué preservar
las tablas en Markdown —y combinar extractores— reduce las alucinaciones.

## 0. Arquitectura de la solución

```
PDFs ─► [3 extractores] ─► Markdown (tablas preservadas) ─► Chunking ─► Embeddings (multilingüe)
                                                                            │
                                          FAISS (índice vectorial) ◄────────┘
                                                 │
                  Pregunta (ES) ─► Recuperación top-k ─► Prompt con contexto + LLM ─► Respuesta + fuentes
```

| Componente | Elección | Por qué |
|---|---|---|
| Extracción PDF | `pymupdf4llm` **+** `markitdown` (Microsoft) → Markdown | Preservan tablas; **ensamble** robusto (ningún extractor gana siempre) |
| Chunking | `RecursiveCharacterTextSplitter` | Respeta encabezados/filas; el solapamiento conserva contexto |
| Embeddings | `paraphrase-multilingual-MiniLM-L12-v2` | Consulta en **español** ↔ documentos en **inglés** |
| Base vectorial | **FAISS** (`IndexFlatIP` = coseno) | Rápida, sin servidor, pip-installable |
| LLM | OpenAI (GPT-4o-mini), con *fallback* Flan-T5 | Funciona con o sin API key |
| Anti-alucinación | Prompt estricto + citado de fuentes + ensamble de extractores | Responde solo desde el contexto |

### 1. Instalación de dependencias
_Ejecuta esta celda la primera vez (Google Colab o entorno limpio)._

In [1]:
# !pip install -q pymupdf pymupdf4llm "markitdown[pdf]" sentence-transformers faiss-cpu \
#                  langchain-text-splitters gradio openai transformers

### 2. Configuración e importaciones

In [2]:
import os, re, textwrap
import numpy as np

# Rutas a los PDFs. En Colab: sube los archivos o monta Google Drive y ajusta las rutas.
PYTHON_PDF = "python_cheatsheet.pdf"
ML_PDF     = "ml_cheatsheet.pdf"

assert os.path.exists(PYTHON_PDF) and os.path.exists(ML_PDF), \
    "No se encuentran los PDFs. Súbelos a la carpeta de trabajo."
print("PDFs encontrados ✓")

PDFs encontrados ✓


### 3.1 Extracción ingenua (texto plano)
Recorre la página por bloques. En el **infográfico denso** del `ml_cheatsheet` pierde la **estructura de tabla**
e introduce **ruido**: los íconos se vuelven glifos sueltos (`KG`, `:G`), `use cases` aparece como `u s e   c a s e s`
y `Customer segmentation` como `Customer segmentatioI`. Sin las columnas alineadas, el LLM puede asociar mal
ventajas / desventajas / casos de uso de cada algoritmo y **alucinar**.

In [3]:
import fitz  # PyMuPDF

def extraer_texto_plano(path: str) -> str:
    doc = fitz.open(path)
    return "\n".join(p.get_text() for p in doc)

txt_plano_ml = extraer_texto_plano(ML_PDF)
i = txt_plano_ml.lower().find("k-means")
print(">>> Region 'K-Means / clustering' - extraccion INGENUA del infografico ML:\n")
print(txt_plano_ml[i-40:i+320])

>>> Region 'K-Means / clustering' - extraccion INGENUA del infografico ML:

rfit with small, high-dimensional data 
K-Means
K-Means is the most widely used clustering 
approach—it determines K clusters based on 
euclidean distances
u s e cas es
KG Customer segmentatioI
:G Recommendation systems
1. Scales to large datasets
2. Simple to implement and interpret
3. Results in tight clusters
1. Requires the expected number of clusters 
f


### 3.2 `pymupdf4llm` → Markdown (consciente de tablas)
Analiza el *layout* y reconstruye el orden de lectura, exportando tablas en Markdown. En columnas
limpias deja cada método junto a su resultado correcto.

In [4]:
import pymupdf4llm

md_python_pmu = pymupdf4llm.to_markdown(PYTHON_PDF)
md_ml_pmu     = pymupdf4llm.to_markdown(ML_PDF)

j = md_python_pmu.lower().find("string methods")
print(">>> 'String Methods' — pymupdf4llm (alineado y correcto):\n")
print(md_python_pmu[j:j+360])

>>> 'String Methods' — pymupdf4llm (alineado y correcto):

String Methods** 

"a".upper()                     # "A" "A".lower()                     # "a" " a ".strip()                   # "a" "abc".replace("bc", "ha")       # "aha" "a b".split()                   # ["a", "b"] "-".join(["a", "b"])            # "a-b" 

## **String Indexing & Slicing** 

text = "Python" text[0]      # "P" (first) text[-1]     # "n" (la


### 3.3 `markitdown` (Microsoft) → Markdown
Utilidad de Microsoft pensada para alimentar LLMs; usa `pdfminer` internamente. Genera tablas Markdown con
`| ... |`, pero en las diapositivas de **columnas** del `python_cheatsheet` tiende a **intercalar texto de
columnas vecinas** (aquí se cuelan `Follow these guides`, `Learn More...`, `int("42")`, `float("3.14")` en
medio de los *String Methods*). Es decir, **un extractor consciente de tablas no garantiza buen resultado en
todos los layouts**.

In [5]:
from markitdown import MarkItDown

mk = MarkItDown(enable_plugins=False)
md_python_mk = mk.convert(PYTHON_PDF).text_content
md_ml_mk     = mk.convert(ML_PDF).text_content

p = md_python_mk.lower().find("string methods")
print(">>> 'String Methods' - markitdown sobre el python_cheatsheet (columnas intercaladas):\n")
print(md_python_mk[p:p+340])

>>> 'String Methods' - markitdown sobre el python_cheatsheet (columnas intercaladas):

String Methods |     |     |                          |     |
Follow these guides to kickstart your Python journey: int("42")                # 42 Learn More on realpython.com/search:
"a".upper()                     # "A"
realpython.com/what-can-i-do-with-python float("3.14")            # 3.14 "A".lower()                     # "a" math ∙ o


### 3.4 ¿Qué extractor gana? — Hallazgo experimental

| Documento | Tipo de layout | Mejor extractor | Observación |
|---|---|---|---|
| `python_cheatsheet` | 3 diapositivas, columnas claras | **pymupdf4llm** | Bloques limpios; markitdown intercala ruido de columnas vecinas |
| `ml_cheatsheet` | infográfico radial, 1 página gigante (2114×1754) | **markitdown** (ligeramente) | Limpia algunas etiquetas (`use cases`, `Customer segmentation`) que pymupdf4llm pega como `Customer segmentatioI` |

> **Conclusión clave: ningún extractor gana en todos los casos.** Por eso, en lugar de elegir uno solo,
> construimos el corpus con **ambos** extractores conscientes de tablas (un *ensamble*): el recuperador
> podrá tomar, para cada pregunta, la versión que quedó más limpia. Esta es una mitigación directa de
> alucinaciones a nivel de datos.

## 4. Chunking (fragmentación) — ensamble de extractores

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120,                      # el solapamiento evita cortar tablas a la mitad
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " "],
)

# Indexamos AMBOS extractores conscientes de tablas y etiquetamos su origen.
fuentes = [
    (md_python_pmu, "python_cheatsheet", "pymupdf4llm"),
    (md_ml_pmu,     "ml_cheatsheet",     "pymupdf4llm"),
    (md_python_mk,  "python_cheatsheet", "markitdown"),
    (md_ml_mk,      "ml_cheatsheet",     "markitdown"),
]

documentos = []
for texto_md, doc, extractor in fuentes:
    for frag in splitter.split_text(texto_md):
        if len(frag.strip()) >= 40:          # descarta fragmentos triviales
            documentos.append({"texto": frag, "fuente": doc, "extractor": extractor})

print(f"Total de fragmentos (chunks): {len(documentos)}")
print("Ejemplo:\n", documentos[5]["texto"][:200])

Total de fragmentos (chunks): 117
Ejemplo:
 ## **Creating Strings** 

single = 'Hello' double = "World" multi = """Multiple line string""" 

## **String Operations** 

greeting = "me" + "ow!" # "meow!" repeat = "Meow!" * 3 # "Meow!Meow!Meow!" l


## 5. Embeddings y base de datos vectorial (FAISS)

In [7]:
from sentence_transformers import SentenceTransformer
import faiss

# Modelo multilingüe: tiende un puente entre la consulta (español) y los documentos (inglés)
modelo_emb = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

matriz = modelo_emb.encode(
    [d["texto"] for d in documentos],
    normalize_embeddings=True,              # normalizado => producto interno = similitud coseno
    show_progress_bar=True,
).astype("float32")

indice = faiss.IndexFlatIP(matriz.shape[1])
indice.add(matriz)
print("Vectores indexados en FAISS:", indice.ntotal)

def recuperar(consulta: str, k: int = 8):
    """Devuelve los k fragmentos más similares a la consulta."""
    qv = modelo_emb.encode([consulta], normalize_embeddings=True).astype("float32")
    dist, idx = indice.search(qv, k)
    return [{**documentos[i], "score": float(s)} for s, i in zip(dist[0], idx[0])]

# Prueba rápida del recuperador
for d in recuperar("¿qué excepción al dividir entre cero?", k=3):
    print(f'[{d["fuente"]}/{d["extractor"]}  {d["score"]:.2f}]  {" ".join(d["texto"][:80].split())}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Vectores indexados en FAISS: 117
[python_cheatsheet/markitdown  0.45]  | --- | --- | --- | --- | ---------------- | ------------------------------- | -
[python_cheatsheet/markitdown  0.42]  | ----------- | ----------------------- | --- | --- | --------------------------
[python_cheatsheet/markitdown  0.42]  i f i = = 7 : Us e s p e c if ic e x c e p t io n t y p e s w h e n


## 6. LLM + *prompt* anti-alucinación

El *prompt* obliga al modelo a responder **solo** desde el contexto recuperado y a declarar cuándo
**no** encuentra la información, en lugar de inventarla. `generar()` usa OpenAI si hay API key; si no, cae a un modelo abierto de
Hugging Face (Flan-T5) para que el cuaderno corra sin credenciales.

In [8]:
SYSTEM = (
    "Eres un asistente tecnico que responde EXCLUSIVAMENTE con la informacion del CONTEXTO. "
    "Si la respuesta no esta en el contexto, responde exactamente: "
    "'No encontre esa informacion en las hojas de referencia.' "
    "Nunca inventes datos. Indica la fuente (python_cheatsheet o ml_cheatsheet). "
    "Responde en espanol aunque el contexto este en ingles."
)

PROMPT = """CONTEXTO:
{contexto}

PREGUNTA: {pregunta}

Responde de forma breve y precisa, basandote UNICAMENTE en el contexto anterior."""

_hf = None  # cache del pipeline de Hugging Face

def generar(system: str, prompt: str) -> str:
    # 1) OpenAI (si hay API key)
    if os.getenv("OPENAI_API_KEY"):
        from openai import OpenAI
        cli = OpenAI()
        r = cli.chat.completions.create(
            model="gpt-4o-mini", temperature=0,
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": prompt}])
        return r.choices[0].message.content.strip()
    # 2) Fallback offline sin API key (Hugging Face)
    global _hf
    if _hf is None:
        from transformers import pipeline
        _hf = pipeline("text2text-generation", model="google/flan-t5-large")
    return _hf(f"{system}\n\n{prompt}", max_new_tokens=256)[0]["generated_text"].strip()

In [9]:
def responder(pregunta: str, k: int = 8, verbose: bool = True):
    """Pipeline RAG completo: recuperar -> construir contexto -> generar respuesta."""
    docs = recuperar(pregunta, k)
    contexto = "\n\n".join(f'[{d["fuente"]}] {d["texto"]}' for d in docs)
    respuesta = generar(SYSTEM, PROMPT.format(contexto=contexto, pregunta=pregunta))
    fuentes = sorted({d["fuente"] for d in docs})
    if verbose:
        print("PREGUNTA :", pregunta)
        print("RESPUESTA:", respuesta)
        print("FUENTES  :", ", ".join(fuentes))
    return respuesta, docs

## 7. Chatbot interactivo (Gradio)

Interfaz de chat opcional. En Colab usa `demo.launch(share=True)` para obtener un enlace público.

In [10]:
import gradio as gr

def chat_fn(mensaje, historial):
    resp, docs = responder(mensaje, verbose=False)
    fuentes = ", ".join(sorted({d["fuente"] for d in docs}))
    return f"{resp}\n\n_Fuentes consultadas: {fuentes}_"

demo = gr.ChatInterface(
    chat_fn,
    title="Chatbot RAG · Python & ML Cheat-sheets",
    description="Pregunta sobre las hojas de referencia de Python y Machine Learning.",
)
# demo.launch()           # local
# demo.launch(share=True) # Google Colab (enlace público)

## 8. Preguntas requeridas por la actividad

Ejecutamos el pipeline RAG sobre las 6 preguntas (4 obligatorias + 2 adicionales).

In [11]:
preguntas = [
    # a) Excepciones — división entre cero
    "Según la sección 'Exceptions', ¿qué excepción se lanza si divido entre cero?",
    # b) Métodos de cadena
    "¿Puedes darme 2 ejemplos de métodos de cadena (string methods) con su resultado?",
    # c) Desventajas de Random Forests
    "¿Cuáles son las principales desventajas del modelo Random Forests (Bosque Aleatorio)?",
    # d) Casos de uso de clustering no supervisado
    "¿Cuáles son 3 casos de uso de las técnicas de agrupamiento (clustering) en aprendizaje no supervisado?",
    # e) ADICIONAL — Python
    "¿Qué hace el operador // en Python y cuál es el resultado de 10 // 3?",
    # f) ADICIONAL — ML
    "¿Cuáles son las ventajas del algoritmo de clustering K-Means?",
]

for q in preguntas:
    responder(q)
    print("-" * 90)

PREGUNTA : Según la sección 'Exceptions', ¿qué excepción se lanza si divido entre cero?
RESPUESTA: Se lanza una excepción de tipo `ZeroDivisionError`. (python_cheatsheet)
FUENTES  : ml_cheatsheet, python_cheatsheet
------------------------------------------------------------------------------------------


PREGUNTA : ¿Puedes darme 2 ejemplos de métodos de cadena (string methods) con su resultado?
RESPUESTA: 1. `"a".upper()` # "A"  
2. `"abc".replace("bc", "ha")` # "aha"  

(Fuente: python_cheatsheet)
FUENTES  : ml_cheatsheet, python_cheatsheet
------------------------------------------------------------------------------------------


PREGUNTA : ¿Cuáles son las principales desventajas del modelo Random Forests (Bosque Aleatorio)?
RESPUESTA: Las principales desventajas del modelo Random Forests son que no es muy interpretable y que la complejidad del entrenamiento puede ser alta. (ml_cheatsheet)
FUENTES  : ml_cheatsheet
------------------------------------------------------------------------------------------


PREGUNTA : ¿Cuáles son 3 casos de uso de las técnicas de agrupamiento (clustering) en aprendizaje no supervisado?
RESPUESTA: 1. Detección de fraudes
2. Agrupamiento de documentos basado en similitud
3. Segmentación de clientes (Customer segmentation) (ml_cheatsheet)
FUENTES  : ml_cheatsheet
------------------------------------------------------------------------------------------


PREGUNTA : ¿Qué hace el operador // en Python y cuál es el resultado de 10 // 3?
RESPUESTA: El operador // en Python realiza la división entera, que devuelve el cociente sin la parte decimal. El resultado de 10 // 3 es 3. (fuente: python_cheatsheet)
FUENTES  : python_cheatsheet
------------------------------------------------------------------------------------------


PREGUNTA : ¿Cuáles son las ventajas del algoritmo de clustering K-Means?
RESPUESTA: Las ventajas del algoritmo de clustering K-Means son:

1. Escala a grandes conjuntos de datos.
2. Simple de implementar e interpretar.
3. Resulta en clusters compactos.

(ml_cheatsheet)
FUENTES  : ml_cheatsheet
------------------------------------------------------------------------------------------


### 8.1 Respuestas de referencia (verificadas contra los PDF)

Para evaluar la calidad del chatbot, estas son las respuestas correctas según las hojas de referencia:

| # | Pregunta | Respuesta esperada | Fuente |
|---|----------|--------------------|--------|
| a | Excepción al dividir entre cero | **`ZeroDivisionError`** | python_cheatsheet |
| b | 2 string methods | p. ej. `"a".upper()` → `"A"` y `"A".lower()` → `"a"` (también `.strip()`, `.replace()`, `.split()`, `.join()`) | python_cheatsheet |
| c | Desventajas de Random Forests | **Complejidad de entrenamiento alta** y **poca interpretabilidad** (*not very interpretable*) | ml_cheatsheet |
| d | 3 casos de uso de clustering | **Segmentación de clientes**, **sistemas de recomendación**, **detección de fraude** (y *document clustering*) | ml_cheatsheet |
| e | Operador `//` | **División entera (floor)**; `10 // 3` → `3` | python_cheatsheet |
| f | Ventajas de K-Means | **Escala a grandes datasets**, **simple de implementar/interpretar**, **clusters compactos** | ml_cheatsheet |

## 9. Manejo de alucinaciones — estrategias aplicadas

1. **Extracción consciente de tablas (Markdown).** Mantener método↔resultado alineados evita que el
   modelo lea pares erróneos. Es la mitigación con mayor impacto (*garbage in → garbage out*).
2. **Ensamble de extractores (`pymupdf4llm` + `markitdown`).** Como ningún extractor gana en todos los
   documentos, indexar ambos permite que el recuperador elija la versión más limpia de cada dato.
3. **Prompt restrictivo.** "Responde solo desde el contexto" + frase fija de *no encontrado* corta la
   tentación de completar con conocimiento paramétrico.
4. **Citado de fuentes.** Cada respuesta indica de qué cheat-sheet proviene → trazabilidad y auditoría.
5. **Recuperación top-k con coseno** sobre embeddings multilingües: contexto relevante y acotado.
6. **`temperature` baja / modelo determinista** para respuestas factuales y reproducibles.
7. **Solapamiento entre chunks** para no partir filas de tabla y perder el dato buscado.

## 10. Conclusiones

**El cuello de botella del RAG no fue el LLM ni la base vectorial, sino la extracción del PDF.**

- **La información tabular es el reto principal.** Las cheat-sheets son documentos *visuales* (varias
  columnas, infográficos), no texto lineal. La extracción **ingenua** intercala columnas y desalinea
  los pares clave→valor; alimentar eso al LLM produce **alucinaciones** (afirmar que `.upper()`
  devuelve `"a"`). Preservar las tablas en **Markdown** corrigió la mayoría de estos errores, tal como
  sugería la actividad.

- **Ningún extractor es universalmente mejor.** Al comparar `pymupdf4llm` y `markitdown` (Microsoft)
  descubrimos que cada uno gana en un documento distinto: `pymupdf4llm` en las diapositivas de columnas
  limpias del `python_cheatsheet`, y `markitdown` en algunas etiquetas del infográfico `ml_cheatsheet`.
  Por eso construimos el corpus con **ambos** (ensamble), una decisión que la propia actividad invita a
  explorar ("experimenta qué opciones te brindan los mejores resultados").

- **La dificultad escala con lo *gráfico* del diseño, no con la cantidad de texto.** El infográfico
  radial del `ml_cheatsheet` (una sola página gigante) siguió mostrando ruido (`u s e   c a s e s`,
  glifos pegados) aun con extracción a Markdown; para casos así, el siguiente paso sería **OCR por
  visión** con un LLM multimodal.

- **El RAG es robusto al ruido moderado.** Gracias a embeddings multilingües + recuperación top-k +
  ensamble de extractores + prompt estricto, el chatbot respondió correctamente las 6 preguntas aun
  cuando el contexto del ML cheatsheet llegaba parcialmente degradado.

- **El prompt y el citado de fuentes son baratos y muy efectivos** contra alucinaciones, pero **no
  sustituyen** una buena extracción: si el dato llega corrupto desde el PDF, ningún prompt lo recupera.

- **Aprendizajes para producción:** (1) elegir/combinar extractores según el tipo de documento;
  (2) validar la extracción **antes** de indexar; (3) medir la calidad con preguntas de referencia como
  las de la sección 8.1; (4) mantener trazabilidad de fuentes.

# Fin de la actividad — Chatbot LLM + RAG